# Mirror Quantum Awesomeness (MQA): Custom Noisy Backend Simulation

This notebook was used to run simulations on a custom noisy backend class.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.providers.fake_provider import GenericBackendV2

from qiskit_device_benchmarking.bench_code.mrb import MirrorQA, QuantumAwesomeness

# custom noisy backend (or an attempt at one, at least)
class NoisyBackend(GenericBackendV2):
    def __init__(
        self,
        num_qubits: int,
        basis_gates: list[str] | None = None,
        coupling_map: list[list[int]] = None,
        p1: float = 0,
        p2: float = 0,
    ):
        self.p = (p1,p2)
        super().__init__(
            num_qubits,
            basis_gates,
            coupling_map=coupling_map,
            noise_info = (p1>0 or p2 >0)
            )
    def _get_noise_defaults(self, name: str, num_qubits: int) -> tuple:
        if name in ['delay', 'reset']:
            return (self.p[0],self.p[0])
        else:
            if num_qubits == 1:
                return (0,0,self.p[0],self.p[0])
            else:
                return (0,0,self.p[1],self.p[1])


In [ ]:
real_device = False

if real_device:
    from qiskit_ibm_provider import IBMProvider
    provider = IBMProvider(instance="ibm-q/open/main")
    backend = provider.get_backend('ibm_sherbrooke')
else:
    p = 0.0015
    backend = NoisyBackend(
    num_qubits=8,
    basis_gates = ["id", "h", "x", "y", "z", "rx", "cx"],
    p1=p/10,
    p2=p,
    )

In [ ]:
# number of shots per circuit
shots = 10000

# lengths of different mirror circuits to run
lengths = [2]+[4,10,20,50,100]

# set up the experiment object
exp = MirrorQA(
    range(backend.num_qubits),
    lengths,
    backend=backend,
    two_qubit_gate_density=0.25,
    num_samples=20,
    initial_entangling_angle=np.pi/4,
    )
exp.set_run_options(
    shots=shots
)


## Run experiment

In [ ]:
#run
rb_data = exp.run()
print(rb_data.job_ids)

## MQA analysis using MRB

In [ ]:
exp.analysis.set_options(analyzed_quantity='Effective Polarization')
#exp.analysis.set_options(analyzed_quantity='Mutual Information')
analysis = exp.analysis.run(rb_data)

In [ ]:
analysis.figure(0)

## MQA analysis using Mutual Information

In [ ]:
qa = QuantumAwesomeness(exp.backend.coupling_map)
mi = qa.mutual_info(rb_data.data())
mmi = qa.mean_mutual_info(rb_data.data(), exp._pairs)

ys = [[[] for _ in range(6)] for _ in range(3)]
yerrs = [[],[],[]]

for p, pairtype in enumerate(['paired', 'unpaired', 'singles']):
    for j, m in enumerate(mmi[pairtype]):
        # m can be array-like; avoid ambiguous truth-value by checking element-wise NaNs
        m_arr = np.asarray(m)
        # skip if all entries are NaN
        if np.all(np.isnan(m_arr)):
            continue
        # use the mean of the available values (ignoring NaNs) as the representative scalar
        val = np.nanmean(m_arr)
        ys[p][j % 6].append(val)

    for j in range(6):
        if len(ys[p][j]) == 0:
            # no data for this bin -> keep a placeholder (nan) and zero error
            yerrs[p].append(0.0)
            ys[p][j] = np.nan
        else:
            yerrs[p].append(np.std(ys[p][j]))
            ys[p][j] = np.mean(ys[p][j])

plt.errorbar(lengths,ys[0],yerr=yerrs[0],label='paired')
plt.errorbar(lengths,ys[1],yerr=yerrs[1],label='spectator')
plt.errorbar(lengths,ys[2],yerr=yerrs[2],label='isolated')
plt.yscale('log')
plt.legend()
plt.xlabel('Circuit Length')
plt.ylabel('Mean Mutual Information')
plt.title('θ = π/2, ρ = 1')